# 01 — Alpaca News Profile

## Purpose
Profile the raw Alpaca news export before transforming it. This notebook should be read-only: do not overwrite or modify the original raw dataset.

## Input
Set `RAW_PATH` to the actual Alpaca CSV path in the repository. Expected source fields include `author`, `content`, `created_at`, `headline`, `id`, `images`, `source`, `summary`, `symbols`, `updated_at`, and `url`.

## Outputs
- Console/profile checks for shape, columns, missingness, duplicates, timestamps, and symbol-list structure.
- Optional profile report saved to `data/processed/news_profile_summary.csv`.

## Important
Coverage has already been validated by the project team. Keep the monthly summary as a reproducible record, but do not infer that an absent row is neutral sentiment or that a month is complete solely because the file loads.

In [ ]:
from pathlib import Path
import ast
import json
import pandas as pd
import numpy as np

# Change this path if your raw file has a different name.
RAW_PATH = Path("../../data/raw/alpaca_news_2021_2025_validated.csv")

if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {RAW_PATH.resolve()}. Update RAW_PATH to your raw Alpaca dataset."
    )

df = pd.read_csv(RAW_PATH, low_memory=False)
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
display(df.head())

Rows: 79,549
Columns: 11


,author,content,created_at,headline,id,images,source,summary,symbols,updated_at,url
0,Henry Khederian,"<p>Every week, Benzinga conducts a sentiment s...",2021-01-02T13:28:07Z,"Will Shopify Stock Reach $2,000 By 2022?",18987669,"[{'size': 'large', 'url': 'https://cdn.benzing...",NaN,"Every week, Benzinga conducts a sentiment surv...","['AMZN', 'BABA', 'ETSY', 'SHOP']",2021-01-02T13:28:08Z,https://www.benzinga.com/trading-ideas/long-id...
1,Gary Anglebrandt,"<p>Tesla Inc delivered 499,550 vehicles in 202...",2021-01-02T16:12:03Z,Tesla Deliveries Fall Just Short Of Musk's 500...,18989549,"[{'size': 'large', 'url': 'https://cdn.benzing...",NaN,"Tesla Inc delivered 499,550 vehicles in 2020, ...",['TSLA'],2021-01-02T16:12:04Z,https://www.benzinga.com/news/21/01/18989549/t...
2,Nelson Hem,<ul>\r\n\t<li><em>Benzinga has examined the pr...,2021-01-02T16:25:30Z,Benzinga's Final Bulls And Bears Of The Year: ...,18989418,"[{'size': 'large', 'url': 'https://cdn.benzing...",NaN,\n\tBenzinga has examined the prospects for ma...,"['AAPL', 'BABA', 'CLR', 'FUBO', 'INTC', 'NIO',...",2021-01-02T16:25:31Z,https://www.benzinga.com/trading-ideas/long-id...
3,Nelson Hem,"<ul>\r\n\t<li><em>This weekend&#39;s <a href=""...",2021-01-02T21:23:47Z,"Barron's First Picks And Pans Of 2021: Disney,...",18990016,"[{'size': 'large', 'url': 'https://cdn.benzing...",NaN,\n\tThis weekend&#39;s Barron&#39;s cover stor...,"['BAC', 'COST', 'CSX', 'DIS', 'EPD', 'FB', 'HD...",2021-01-02T21:23:47Z,https://www.benzinga.com/media/21/01/18990016/...
4,Shanthi Rexaline,<p>Chinese EV maker <strong>Nio Inc &ndash; AD...,2021-01-03T13:12:08Z,Nio's Record Run Continues as Deliveries Spike...,18990977,"[{'size': 'large', 'url': 'https://cdn.benzing...",NaN,Chinese EV maker Nio Inc – ADR (NYSE: NIO) ann...,"['LI', 'NIO', 'TSLA']",2021-01-03T13:12:08Z,https://www.benzinga.com/news/21/01/18990977/n...


## 1. Inspect the schema and data types

Check whether the actual file matches the expected Alpaca export. Do not rename columns in the raw DataFrame; renaming belongs in the cleaning notebook.

In [2]:
expected_columns = {
    "author", "content", "created_at", "headline", "id", "images",
    "source", "summary", "symbols", "updated_at", "url"
}
missing_columns = sorted(expected_columns - set(df.columns))
extra_columns = sorted(set(df.columns) - expected_columns)

print("Missing expected columns:", missing_columns or "None")
print("Additional columns:", extra_columns or "None")
display(pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "null_count": df.isna().sum(),
    "null_pct": (df.isna().mean() * 100).round(2)
}))

Missing expected columns: None
Additional columns: None


,dtype,non_null,null_count,null_pct
author,str,79549,0,0.00
content,str,57744,21805,27.41
created_at,str,79549,0,0.00
headline,str,79549,0,0.00
id,int64,79549,0,0.00
images,str,79549,0,0.00
source,str,65035,14514,18.25
summary,str,57712,21837,27.45
symbols,str,79549,0,0.00
updated_at,str,79549,0,0.00


## 2. Check article IDs and duplicates

Alpaca `id` is the source identifier. Do not use a generated row number as the article identity. If duplicate IDs exist, inspect them before deciding which record to retain.

In [3]:
if "id" in df.columns:
    print("Missing article IDs:", int(df["id"].isna().sum()))
    print("Unique non-null IDs:", int(df["id"].nunique(dropna=True)))
    print("Rows in duplicate-ID groups:", int(df["id"].duplicated(keep=False).sum()))
    duplicate_ids = df[df["id"].duplicated(keep=False)].sort_values("id")
    display(duplicate_ids.head(20))

Missing article IDs: 0
Unique non-null IDs: 79549
Rows in duplicate-ID groups: 0


,author,content,created_at,headline,id,images,source,summary,symbols,updated_at,url


## 3. Parse timestamps for inspection

This is an inspection only. The cleaned notebook will preserve timezone-aware UTC timestamps and will not overwrite the raw strings.

In [4]:
for col in ["created_at", "updated_at"]:
    if col in df.columns:
        parsed = pd.to_datetime(df[col], utc=True, errors="coerce")
        print(f"\n{col}")
        print("Unparseable / missing:", int(parsed.isna().sum()))
        print("Earliest:", parsed.min())
        print("Latest:", parsed.max())


created_at
Unparseable / missing: 0
Earliest: 2021-01-02 13:28:07+00:00
Latest: 2025-12-31 22:30:46+00:00

updated_at
Unparseable / missing: 0
Earliest: 2021-01-02 13:28:08+00:00
Latest: 2025-12-31 22:30:46+00:00


## 4. Inspect `symbols`

Alpaca may store the symbols field as a Python-list-like string, a list, or a missing value. The next notebook handles these cases. Here, profile the number of symbols per article without modifying the source data.

In [5]:
def parse_symbols_for_profile(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    if isinstance(value, (list, tuple, set)):
        items = list(value)
    elif isinstance(value, str):
        value = value.strip()
        if not value:
            return []
        try:
            parsed = ast.literal_eval(value)
            items = list(parsed) if isinstance(parsed, (list, tuple, set)) else [value]
        except (ValueError, SyntaxError):
            # Conservative fallback for a simple comma-separated string.
            items = value.split(",")
    else:
        return []
    return [str(x).strip().upper() for x in items if str(x).strip()]

if "symbols" in df.columns:
    symbol_lists = df["symbols"].apply(parse_symbols_for_profile)
    symbol_counts = symbol_lists.apply(len)
    print(symbol_counts.describe())
    display(symbol_counts.value_counts().sort_index().rename_axis("symbol_count").to_frame("article_count"))
    print("Rows with no parsed symbols:", int((symbol_counts == 0).sum()))

count    79549.000000
mean         4.762323
std         16.016856
min          1.000000
25%          1.000000
50%          2.000000
75%          5.000000
max        856.000000
Name: symbols, dtype: float64


,article_count
symbol_count,
1,31497
2,12156
3,7721
4,5871
5,4770
...,...
756,1
792,1
838,1


Rows with no parsed symbols: 0


## 5. Monthly row counts

This is a reproducible audit table, not proof by itself that source coverage is complete. Compare the output with the coverage validation already performed by the team.

In [6]:
if "created_at" in df.columns:
    created = pd.to_datetime(df["created_at"], utc=True, errors="coerce")
    monthly = (
        created.dropna().dt.to_period("M").astype(str)
        .value_counts().sort_index().rename_axis("month").reset_index(name="article_count")
    )
    display(monthly)
else:
    monthly = pd.DataFrame(columns=["month", "article_count"])

C:\Users\bosya\AppData\Local\Temp\ipykernel_32348\3255400910.py:4: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  created.dropna().dt.to_period("M").astype(str)


,month,article_count
0,2021-01,902
1,2021-02,926
2,2021-03,1058
3,2021-04,1509
4,2021-05,976
5,2021-06,1273
6,2021-07,1493
7,2021-08,1245
8,2021-09,1313
9,2021-10,1407


## 6. Generate the News Data Dictionary

The raw dataset remains unchanged. This step generates `data/processed/news_data_dictionary.md` to document the dataset's structure and basic quality checks.

The data dictionary includes:
- Dataset dimensions and profiling statistics.
- Column names, data types, and descriptions.
- Missing-value and unique-value counts for each column.
- Data quality observations and recommendations for downstream processing.

This document supports project documentation and helps the team understand the raw Alpaca news data before cleaning, sentiment analysis, and ticker mapping.


In [7]:

# Create the output directory
output_dir = Path("../../docs/methodology")
output_dir.mkdir(parents=True, exist_ok=True)

# Collect dataset-level statistics
summary = pd.DataFrame([
    {"metric": "raw_rows", "value": len(df)},
    {"metric": "raw_columns", "value": len(df.columns)},
    {
        "metric": "unique_article_ids",
        "value": df["id"].nunique(dropna=True) if "id" in df.columns else np.nan
    },
    {
        "metric": "duplicate_id_rows",
        "value": int(df["id"].duplicated(keep=False).sum())
        if "id" in df.columns else np.nan
    },
    {
        "metric": "missing_headline_rows",
        "value": int(df["headline"].isna().sum())
        if "headline" in df.columns else np.nan
    },
    {
        "metric": "missing_created_at_rows",
        "value": int(df["created_at"].isna().sum())
        if "created_at" in df.columns else np.nan
    },
])

# Output file
PROFILE_OUT = output_dir / "news_data_dictionary.md"

# Start building the Markdown document
lines = [
    "# MarketPulse — Alpaca News Data Dictionary",
    "",
    "## 1. Dataset Overview",
    "",
    "- **Source:** Alpaca News API",
    "- **Dataset type:** Raw financial news articles",
    f"- **Total rows:** {len(df):,}",
    f"- **Total columns:** {len(df.columns)}",
    "",
    "## 2. Dataset Profile",
    "",
    "| Metric | Value |",
    "|---|---:|",
]

for _, row in summary.iterrows():
    value = row["value"]
    if pd.isna(value):
        value = "N/A"
    elif isinstance(value, (int, np.integer)):
        value = f"{value:,}"
    lines.append(f"| {row['metric']} | {value} |")

lines += [
    "",
    "## 3. Column Definitions",
    "",
    "| Column | Data Type | Missing Values | Unique Values | Description |",
    "|---|---|---:|---:|---|",
]

# Descriptions for known Alpaca fields
descriptions = {
    "id": "Unique identifier assigned to the news article by Alpaca.",
    "author": "Author or publisher attribution provided by the source.",
    "content": "Article content provided by the API; may be incomplete or absent.",
    "created_at": "Timestamp indicating when the article was created or published, as provided by Alpaca.",
    "headline": "Headline or title of the news article.",
    "images": "Image metadata associated with the article, when available.",
    "source": "Publisher or source identifier for the article.",
    "summary": "Summary text provided for the article, when available.",
    "symbols": "Ticker symbols associated with the article; an article may have multiple symbols.",
    "updated_at": "Timestamp indicating the article's last update, when provided.",
    "url": "Link to the original news article."
}

for column in df.columns:
    dtype = str(df[column].dtype)
    missing = int(df[column].isna().sum())
    unique = int(df[column].nunique(dropna=True))
    description = descriptions.get(
        column,
        "Description needs to be confirmed from the source schema."
    )

    # Prevent Markdown table formatting issues
    column_name = str(column).replace("|", "\\|")
    description = description.replace("|", "\\|").replace("\n", " ")

    lines.append(
        f"| `{column_name}` | {dtype} | {missing:,} | "
        f"{unique:,} | {description} |"
    )

lines += [
    "",
    "## 4. Data Quality Notes",
    "",
    "- Check duplicate article IDs before downstream processing.",
    "- Check missing headlines and timestamps.",
    "- Parse timestamps into a consistent timezone-aware format; preserve UTC.",
    "- Parse the `symbols` field into a structured list before ticker mapping.",
    "- Clean HTML and whitespace from text fields before sentiment analysis.",
    "- Do not assume missing article content means the article has no useful information; "
    "the headline and summary may still be available.",
    "",
    "## 5. Downstream Use in MarketPulse",
    "",
    "- Deduplicate articles using the original article ID.",
    "- Apply FinBERT once per unique article.",
    "- Keep article-level sentiment separate from article–ticker associations.",
    "- Filter ticker associations using the team's agreed project ticker list.",
    "- Document any transformations performed during cleaning.",
    "",
    "---",
    "",
    "*Generated automatically from the raw Alpaca news DataFrame.*",
]

# Save the Markdown document
PROFILE_OUT.write_text("\n".join(lines) + "\n", encoding="utf-8")

print(f"Saved data dictionary to: {PROFILE_OUT.resolve()}")
print(f"Documented {len(df.columns)} columns.")
display(summary)


Saved data dictionary to: D:\DEPI\MarketPulse Graduation Project\docs\methodology\news_data_dictionary.md
Documented 11 columns.


,metric,value
0,raw_rows,79549
1,raw_columns,11
2,unique_article_ids,79549
3,duplicate_id_rows,0
4,missing_headline_rows,0
5,missing_created_at_rows,0


## Completion checklist

- [ ] Confirm the raw path points to the validated Alpaca file.
- [ ] Review duplicate IDs and missing identifiers.
- [ ] Review unparseable timestamps.
- [ ] Review the distribution of symbols per article.
- [ ] Record any caveats in the project's data-quality notes.
- [ ] Do not edit or overwrite the raw dataset.